# Módulo 00 — internals do modelo: a base cheia

O [walkthrough](srag_model_walkthrough.ipynb) treina na amostra commitada.
Este caderno paga a conta completa — **onde a base cheia existe**: o Ouro
em disco (`~/Documents/srag-data/gold/`). Reconstruí-lo do Bronze estoura a
memória de um runner de CI (o `S.build` materializa 1,7 M × 420 no pior
ano — OOM medido em 2026-09-01, o modo de falha que o próprio módulo 00
documenta). Em ambiente sem o Ouro, cada seção pesada **imprime o pulo
declarado** em vez de fingir; os outputs commitados vêm de uma execução
local completa, e é isso que a barra de evidência exige.

In [1]:
%pip install -q pandas pyarrow scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pathlib
import subprocess
import sys
import time

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

REPO = pathlib.Path.cwd()
while REPO != REPO.parent and not (REPO / "tools" / "srag_model.py").exists():
    REPO = REPO.parent
if not (REPO / "tools" / "srag_model.py").exists():
    import tempfile
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                        str(REPO)], check=True)
sys.path.insert(0, str(REPO / "tools"))
import srag_model as M

OURO_PATH = pathlib.Path.home() / "Documents/srag-data/gold/gold_covid_obito.parquet"
if OURO_PATH.exists():
    ouro = pd.read_parquet(OURO_PATH)
    for c in M.CATEGORICAS:
        ouro[c] = ouro[c].astype("category")
    print(f"Ouro: {len(ouro):,} × {len(ouro.columns)}")
    print(ouro.groupby("split", observed=True)["y_obito"]
          .agg(["size", "sum", "mean"]).round(3).to_string())
else:
    ouro = None
    print("SEM Ouro local — as seções de base cheia imprimem o pulo declarado;")
    print("os outputs commitados vêm de uma execução local completa.")

Ouro: 1,282,970 × 50
          size     sum   mean
split                        
test     16142    2945  0.182
train  1242680  390769  0.314
val      24148    4572  0.189


## 1. O preço da amostra: fit cheio vs fit de 200 mil

O delta é o número que autoriza o desenho do curso: se for pequeno,
explicar o modelo-da-amostra é explicar o modelo. A avaliação é idêntica
nos dois (val e teste inteiros, por construção da amostra).

In [3]:
amostra = M.load_gold(REPO / "modules/00-dataset/gold/gold_covid_obito_sample.parquet")
t0 = time.time()
xgb_amostra, _ = M.fit_models(amostra)
t_amostra = time.time() - t0

pares = [("amostra (200 k)", xgb_amostra, amostra, t_amostra)]
xgb_cheio = None
if ouro is not None:
    t0 = time.time()
    xgb_cheio, _logit_cheio = M.fit_models(ouro)
    pares.insert(0, ("cheio (1,24 M)", xgb_cheio, ouro, time.time() - t0))
else:
    print("PULADO (fit cheio): sem Ouro local.")

linhas = []
for nome, modelo, base, t in pares:
    for s in ("val", "test"):
        linhas.append({"treino": nome, "fit_s": round(t, 1)} | M.metrics(modelo, base, s))
print(pd.DataFrame(linhas).to_string(index=False))

         treino  fit_s split     n  obitos    auc  brier  previsto_medio  observado
 cheio (1,24 M)   26.4   val 24148    4572 0.7601 0.1355          0.2253     0.1893
 cheio (1,24 M)   26.4  test 16142    2945 0.7680 0.1304          0.2196     0.1824
amostra (200 k)    4.1   val 24148    4572 0.7564 0.1356          0.2209     0.1893
amostra (200 k)    4.1  test 16142    2945 0.7644 0.1308          0.2154     0.1824


## 2. AUC por ano de início — a deriva, medida onde dói

In [4]:
from sklearn.metrics import roc_auc_score

if ouro is None:
    print("PULADO: exige a base cheia (outputs commitados carregam a tabela).")
else:
    x_xgb, _, y = M.design_matrix(ouro)
    p = xgb_cheio.predict_proba(x_xgb)[:, 1]
    por_ano = []
    for ano in sorted(ouro["ano_onset"].unique()):
        m = (ouro["ano_onset"] == ano).to_numpy()
        if ouro.loc[m, "y_obito"].nunique() < 2:
            continue
        por_ano.append({
            "ano": int(ano), "n": int(m.sum()),
            "CFR %": round(100 * float(ouro.loc[m, "y_obito"].mean()), 1),
            "AUC": round(float(roc_auc_score(y[m], p[m])), 4),
            "previsto médio": round(float(p[m].mean()), 3),
        })
    print(pd.DataFrame(por_ano).to_string(index=False))

 ano      n  CFR %    AUC  previsto médio
2020 407679   32.2 0.7890           0.322
2021 723037   31.4 0.7546           0.314
2022 111964   28.8 0.7451           0.289
2023  24148   18.9 0.7601           0.225
2024  16142   18.2 0.7680           0.220


A discriminação (AUC) resiste melhor que a calibração: o ranking de risco
viaja entre regimes; o nível absoluto, não.

## 3. Curva de aprendizado — quanto treino compra o quê

In [5]:
if ouro is None:
    print("PULADO: exige a base cheia.")
else:
    rng = np.random.default_rng(42)
    treino_cheio = ouro[ouro["split"] == "train"].reset_index(drop=True)
    aval = ouro[ouro["split"] != "train"]
    curva = []
    for n_treino in (50_000, 200_000, len(treino_cheio)):
        if n_treino < len(treino_cheio):
            idx = np.sort(rng.choice(len(treino_cheio), size=n_treino, replace=False))
            base = pd.concat([treino_cheio.iloc[idx], aval], ignore_index=True)
        else:
            base = ouro
        for c in M.CATEGORICAS:
            base[c] = base[c].astype("category")
        mdl, _ = M.fit_models(base)
        curva.append({"treino": n_treino} | {
            k: v for k, v in M.metrics(mdl, base, "test").items()
            if k in ("auc", "brier")
        })
    print(pd.DataFrame(curva).to_string(index=False))

 treino    auc  brier
  50000 0.7484 0.1331
 200000 0.7656 0.1307
1242680 0.7680 0.1304


## 4. O exemplar sobrevive ao modelo cheio?

In [6]:
ex_amostra = M.pick_exemplar(xgb_amostra, amostra)
print(f"regra na amostra   → gold_id {int(ex_amostra.gold_id)}  p = {ex_amostra.p_obito:.3f}")
if ouro is None:
    print("PULADO (comparação com o modelo cheio): sem Ouro local.")
else:
    ex_cheio = M.pick_exemplar(xgb_cheio, ouro)
    print(f"regra na base cheia → gold_id {int(ex_cheio.gold_id)}  p = {ex_cheio.p_obito:.3f}")
    p_do_exemplar = float(M.predict_proba(
        xgb_cheio, ouro[ouro["gold_id"] == int(ex_amostra.gold_id)]
    )[0])
    print(f"p do exemplar-da-amostra sob o modelo cheio: {p_do_exemplar:.3f}")
print("\nO paciente do curso é o da AMOSTRA — o objeto de estudo dos módulos.")
print("Se as regras divergem entre amostra e base, é registro, não problema.")

regra na amostra   → gold_id 1276776  p = 0.500
regra na base cheia → gold_id 1270631  p = 0.500
p do exemplar-da-amostra sob o modelo cheio: 0.441

O paciente do curso é o da AMOSTRA — o objeto de estudo dos módulos.
Se as regras divergem entre amostra e base, é registro, não problema.


## 5. As quatro impossibilidades: amostra vs base cheia

A regra do repositório: medir no cheio E re-medir na amostra, imprimir os
dois — diferença além de composição é achado sobre a amostra.

In [7]:
def fracoes(base):
    n = len(base)
    return {
        "portão do funil": 100 * int((~base["fator_risc_portao"].astype(bool)).sum()) / n,
        "pré-campanha": 100 * int((base["meses_desde_mar2020"] < M.MESES_CAMPANHA).sum()) / n,
        "critério-2 por um fio": 100 * int((base["n_crit2"] == 1).sum()) / n,
        "critério-3 por um fio": 100 * int((base["n_crit3"] == 1).sum()) / n,
    }

colunas = {"amostra %": fracoes(amostra)}
if ouro is not None:
    colunas = {"base cheia %": fracoes(ouro)} | colunas
quadro = pd.DataFrame(colunas).round(2)
if "base cheia %" in quadro:
    quadro["Δ"] = (quadro["amostra %"] - quadro["base cheia %"]).round(2)
print(quadro.to_string())
print("\nA amostra sub-pesa o pré-campanha porque val/test (2023-24, inteiros)")
print("pesam mais nela — efeito de composição declarado; os módulos citam a")
print("fração da base em que operam.")

                       base cheia %  amostra %     Δ
portão do funil               38.05      37.27 -0.78
pré-campanha                  34.50      29.67 -4.83
critério-2 por um fio         78.83      79.60  0.77
critério-3 por um fio         23.66      24.67  1.01

A amostra sub-pesa o pré-campanha porque val/test (2023-24, inteiros)
pesam mais nela — efeito de composição declarado; os módulos citam a
fração da base em que operam.
